In [1]:
"""
- ID3 (Iterative Dichotomiser 3) was developed in 1986 by Ross Quinlan. The algorithm creates a multiway tree, finding 
      for each node (i.e. in a greedy manner) the categorical feature that will yield the largest information gain for 
      categorical targets. Trees are grown to their maximum size and then a pruning step is usually applied to improve 
      the ability of the tree to generalise to unseen data.

- C4.5 is the successor to ID3 and removed the restriction that features must be categorical by dynamically defining a 
      discrete attribute (based on numerical variables) that partitions the continuous attribute value into a discrete 
      set of intervals. C4.5 converts the trained trees (i.e. the output of the ID3 algorithm) into sets of if-then rules.
      These accuracy of each rule is then evaluated to determine the order in which they should be applied. Pruning is 
      done by removing a rule’s precondition if the accuracy of the rule improves without it.

- C5.0 is Quinlan’s latest version release under a proprietary license. It uses less memory and builds smaller rulesets 
      than C4.5 while being more accurate.

- CART (Classification and Regression Trees) is very similar to C4.5, but it differs in that it supports numerical 
      target variables (regression) and does not compute rule sets. CART constructs binary trees using the feature and 
      threshold that yield the largest information gain at each node.

*** scikit-learn uses an optimised version of the CART algorithm ***
Documentation: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
"""

%matplotlib inline
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("error", category=UndefinedMetricWarning)
warnings.filterwarnings(action="ignore",category=DeprecationWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_validate
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer
from sklearn.metrics import precision_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import fbeta_score
from sklearn.metrics import auc
from sklearn.model_selection import StratifiedKFold
from sklearn import metrics
from sklearn.tree import export_graphviz
from sklearn.externals.six import StringIO  
from IPython.display import Image  
from collections import OrderedDict
import pydotplus
import pickle


def PlotTree(clf, features, name="Tree.png"):
    dot_data=StringIO()
    export_graphviz(clf, out_file=dot_data, filled=True, rounded=True, special_characters=True,
                    feature_names=features,class_names=['0.0','1.0'])
    graph = pydotplus.graph_from_dot_data(dot_data.getvalue())  
    graph.write_png(name)
    return Image(graph.create_png())

def GetMetrics(y_test, y_pred):
    print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
    display("confusion_matrix", pd.DataFrame(metrics.confusion_matrix(y_test, y_pred)))
    report = metrics.classification_report(y_test, y_pred, output_dict=True)
    display("metrics", pd.DataFrame(report).transpose())



In [2]:
## Load Dataset ....

Class=["refatoracao"]
features = [ "LOC", "NPM", "WMC", "NOC", "DIT", "CAM", 
             "CBO", "RFC", "CA", "CE", "LCOM3", #"LCOM",
             "DAM", "MOA", "MFA", "IC", "CBM", "AMC", 
           ]
df = pd.read_csv('dataset_balanced_refat_final.csv',delimiter=";", decimal=".")
x = df[features]
y = df[Class]
print("Data Size: ",df.shape, 
      "Class 1.0: ", len(df[df.refatoracao==1.0]), 
      "Class 0.0: ", len(df[df.refatoracao!=1.0]), )
df[features+Class].head()

Data Size:  (2016, 23) Class 1.0:  1008 Class 0.0:  1008


,LOC,NPM,WMC,NOC,DIT,CAM,CBO,RFC,CA,CE,LCOM3,DAM,MOA,MFA,IC,CBM,AMC,refatoracao
0,1656.0,88.0,100.0,0.0,0.0,0.2148,6.0,140.0,0.0,6.0,0.2222,0.8333,0.0,0.0,0.0,0.0,15.5,1.0
1,253.0,11.0,14.0,4.0,1.0,0.1688,7.0,46.0,4.0,3.0,0.8718,0.3333,0.0,0.0,0.0,0.0,168571.0,1.0
2,1421.0,88.0,96.0,0.0,0.0,0.2211,6.0,130.0,0.0,6.0,0.5263,0.5000,0.0,0.0,0.0,0.0,137813.0,1.0
3,517.0,24.0,30.0,0.0,0.0,0.2867,5.0,57.0,0.0,5.0,0.9151,0.0000,0.0,0.0,0.0,0.0,15.8,1.0
4,783.0,17.0,22.0,0.0,0.0,0.2273,8.0,65.0,0.0,8.0,0.9568,0.1875,1.0,0.0,0.0,0.0,331364.0,1.0


In [3]:
## Find the best params for DecisionTreeClassifier

dec_tree = DecisionTreeClassifier(random_state=0) # random_state - deterministic behaviour during fitting random_state has to be fixed to an integer
pipe = Pipeline(steps=[ ('dec_tree', dec_tree),
                      ])
parameters = dict(dec_tree__criterion=['gini', 'entropy'],
                  dec_tree__min_samples_split=[16, 32, 64, 128, 256],
                  dec_tree__min_samples_leaf=[16, 32, 64, 128, 256],
                  dec_tree__max_depth=[2, 4, 6, 8, 10, 12, 14, 16, 18],
                  dec_tree__max_features=list(range(1,len(features)+1)),
                 )
scores = {'accuracy' :make_scorer(accuracy_score),
          'recall'   :make_scorer(recall_score),
          'precision':make_scorer(precision_score),
          'f1'       :make_scorer(fbeta_score, beta = 1),
          'auc'      :make_scorer(auc),
         }
KF = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
clf_GS = GridSearchCV(pipe, param_grid=parameters, n_jobs=-1, cv=KF,
                      #scoring = scores,   refit = 'auc', 
                      scoring = 'roc_auc',
                     )

# split in Train_Test (70%) and Validation (30%) (train, test, validation, prevent overfitting)
x_train_test, x_validation, y_train_test, y_validation = train_test_split(x, y, test_size=0.3, random_state=0) 
# KFold will split Train_Test into 10-Folds: 9 to Train and 1 to Test. Next, GridSearchCV will find the best model
clf_GS_fit=clf_GS.fit(x_train_test, y_train_test) 

pickle.dump(clf_GS, open("BestDecisionTree", 'wb')) # Save Model

print("Decision tree tuning")
print("Steps:\n",pipe.steps)
print("Best Parameters:\n\t Params: {}\n\t Score: {}\n".format(clf_GS.best_params_,clf_GS.best_score_))

print("** metrics for Validation dataset")
y_pred = clf_GS_fit.predict(x_validation)
GetMetrics(y_validation, y_pred)

img=PlotTree(clf_GS.best_estimator_.named_steps["dec_tree"], features, "Tree_GS.png")


Decision tree tuning
Steps:
 [('dec_tree', DecisionTreeClassifier(class_weight=None, criterion='gini', max_depth=None,
            max_features=None, max_leaf_nodes=None,
            min_impurity_decrease=0.0, min_impurity_split=None,
            min_samples_leaf=1, min_samples_split=2,
            min_weight_fraction_leaf=0.0, presort=False, random_state=0,
            splitter='best'))]
Best Parameters:
	 Params: {'dec_tree__criterion': 'entropy', 'dec_tree__max_depth': 8, 'dec_tree__max_features': 7, 'dec_tree__min_samples_leaf': 16, 'dec_tree__min_samples_split': 64}
	 Score: 0.9302322795739711

** metrics for Validation data set
Accuracy: 0.8446280991735537


'confusion_matrix'

,0,1
0,250,56
1,38,261


'metrics'

,f1-score,precision,recall,support
0.0,0.841751,0.868056,0.816993,306.0
1.0,0.847403,0.823344,0.872910,299.0
micro avg,0.844628,0.844628,0.844628,605.0
macro avg,0.844577,0.845700,0.844952,605.0
weighted avg,0.844544,0.845958,0.844628,605.0


In [4]:
TrainParmResults=pd.DataFrame(clf_GS.cv_results_)
pd.options.display.max_columns = None
TrainParmResults.head(6) # -1 Show all 

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_dec_tree__criterion,param_dec_tree__max_depth,param_dec_tree__max_features,param_dec_tree__min_samples_leaf,param_dec_tree__min_samples_split,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,split5_train_score,split6_train_score,split7_train_score,split8_train_score,split9_train_score,mean_train_score,std_train_score
0,0.005225,0.004184,0.004911,0.002325,gini,2,1,16,16,"{'dec_tree__criterion': 'gini', 'dec_tree__max...",0.655921,0.651855,0.626358,0.639034,0.673642,0.649698,0.648893,0.531992,0.712475,0.665612,0.645546,0.043746,7455,0.663139,0.663628,0.643831,0.664993,0.661273,0.641200,0.641335,0.676835,0.656889,0.662068,0.657519,0.011174
1,0.006270,0.003549,0.005005,0.006041,gini,2,1,16,32,"{'dec_tree__criterion': 'gini', 'dec_tree__max...",0.655921,0.651855,0.626358,0.639034,0.673642,0.649698,0.648893,0.531992,0.712475,0.665612,0.645546,0.043746,7455,0.663139,0.663628,0.643831,0.664993,0.661273,0.641200,0.641335,0.676835,0.656889,0.662068,0.657519,0.011174
2,0.004346,0.004100,0.002572,0.003160,gini,2,1,16,64,"{'dec_tree__criterion': 'gini', 'dec_tree__max...",0.655921,0.651855,0.626358,0.639034,0.673642,0.649698,0.648893,0.531992,0.712475,0.665612,0.645546,0.043746,7455,0.663139,0.663628,0.643831,0.664993,0.661273,0.641200,0.641335,0.676835,0.656889,0.662068,0.657519,0.011174
3,0.005053,0.003503,0.001933,0.002089,gini,2,1,16,128,"{'dec_tree__criterion': 'gini', 'dec_tree__max...",0.655921,0.651855,0.626358,0.639034,0.673642,0.649698,0.648893,0.531992,0.712475,0.665612,0.645546,0.043746,7455,0.663139,0.663628,0.643831,0.664993,0.661273,0.641200,0.641335,0.676835,0.656889,0.662068,0.657519,0.011174
4,0.001609,0.002777,0.002715,0.003565,gini,2,1,16,256,"{'dec_tree__criterion': 'gini', 'dec_tree__max...",0.655921,0.651855,0.626358,0.639034,0.673642,0.649698,0.648893,0.531992,0.712475,0.665612,0.645546,0.043746,7455,0.663139,0.663628,0.643831,0.664993,0.661273,0.641200,0.641335,0.676835,0.656889,0.662068,0.657519,0.011174
5,0.002656,0.002446,0.002965,0.003285,gini,2,1,32,16,"{'dec_tree__criterion': 'gini', 'dec_tree__max...",0.655227,0.628546,0.667304,0.646579,0.649195,0.699396,0.605131,0.552414,0.692656,0.649490,0.644587,0.040366,7460,0.660323,0.665051,0.662438,0.662176,0.666604,0.656885,0.654120,0.676498,0.654742,0.665442,0.662428,0.006283


In [6]:
# https://sklearn-template.readthedocs.io/en/latest/user_guide.html
class PaperTreeModel(BaseEstimator, ClassifierMixin):
    def __init__(self):
        pass

    def fit(self, x, y): #Pass, do not create a model.....
        return self
    
    def CreateOut(self, data, out):
        if len(data)>0:
            index=data.index.values.tolist()
            tmp=dict.fromkeys(index , out)
            self.out_={**self.out_, **tmp}
        
    def predict(self, x): # Paper model
        self.out_={}
        self.CreateOut(x[ (x["WMC"] >9.500) ], 1.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]>597) & (x["WMC"]>8.50) ], 1.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]>597) & (x["WMC"]<=8.50) ], 0.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]>7.50) & (x["CAM"]>0.270) & (x["NOC"]>0.50) & (x["NOC"]>1.50) ], 0.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]>7.50) & (x["CAM"]>0.270) & (x["NOC"]>0.50) & (x["NOC"]<=1.50) ], 1.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]>7.50) & (x["CAM"]>0.270) & (x["NOC"]<=0.50) & (x["LOC"]>48) ], 0.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]>7.50) & (x["CAM"]>0.270) & (x["NOC"]<=0.50) & (x["LOC"]<=48) & (x["DIT"]>0.50) ], 0.0)        
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]>7.50) & (x["CAM"]>0.270) & (x["NOC"]<=0.50) & (x["LOC"]<=48) & (x["DIT"]<=0.50) ], 1.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]>7.50) & (x["CAM"]>0.270) ], 1.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]<=7.50) ], 0.0)
        self.CreateOut(x[~x.index.isin(list(self.out_.keys()))], 1.0 )
        self.out_ = OrderedDict(sorted(self.out_.items()))
        return list(self.out_.values())


# Test Paper model ....
clf=PaperTreeModel()
KF = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
scores = cross_validate(clf, x, y, cv=KF, scoring=['precision', 'recall', 'f1'])
display(pd.DataFrame(scores))

,fit_time,score_time,test_precision,train_precision,test_recall,train_recall,test_f1,train_f1
0,0.000000,0.050286,0.787402,0.764505,0.990099,0.987872,0.877193,0.861953
1,0.000000,0.050066,0.773438,0.766012,0.980198,0.988975,0.864629,0.863330
2,0.000000,0.049941,0.787402,0.764505,0.990099,0.987872,0.877193,0.861953
3,0.000000,0.049765,0.765152,0.766924,1.000000,0.986770,0.866953,0.863067
4,0.000000,0.051114,0.751938,0.768376,0.960396,0.991180,0.843478,0.865672
5,0.004118,0.039268,0.748148,0.768900,1.000000,0.986770,0.855932,0.864317
6,0.000000,0.046028,0.793651,0.763853,0.990099,0.987872,0.881057,0.861538
7,0.006589,0.047596,0.742647,0.769561,1.000000,0.986770,0.852321,0.864734
8,0.000000,0.051381,0.790323,0.764255,0.980000,0.988987,0.875000,0.862218
9,0.000000,0.053073,0.733333,0.770619,0.990000,0.987885,0.842553,0.865830
